In [1]:
%load_ext autoreload
%autoreload 1
%aimport api_checks.model_calculator
%aimport api_checks.api_cache
%aimport api_checks.model_parmeters

c:\Users\wildn\Desktop\New life\AI\Expirments\VisulaiztionInfoFlowDemo\real\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

from api_checks.api_cache import APICache
from info_flow.config import Config
import torch
config = Config()
api_cache = APICache(hf_token=config.hf_token, cache_path=Path(config.result_cache_path))
full_run_result = api_cache.get_full_run_results(config.info_flow_model, "First test, lets see")
original_logi = full_run_result.logits
last_mlp_output = full_run_result.contributions.post_mlp_contribution[-1].sum(dim=1) #(p_len,d_model)
new_logits = api_cache.get_infomration_calculator(config.info_flow_model).calc_logits(last_mlp_output)
print(new_logits.dtype)
print(original_logi.dtype)


c:\Users\wildn\Desktop\New life\AI\Expirments\VisulaiztionInfoFlowDemo\real\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CACHE HIT meta-llama/Llama-3.1-8B First test, lets see 2026-06-30 09:59:27.901336


⬇ Downloading: 100%|██████████| 4.31k/4.31k [00:00<00:00]


torch.Size([6, 4096])
torch.Size([6, 1])
torch.float32
torch.bfloat16


In [3]:
print((new_logits.norm()))
print((original_logi.norm()))
original_logi = original_logi.to(torch.bfloat16)
print((new_logits-original_logi).norm())
print(torch.abs(new_logits-original_logi).max())
diff = new_logits-original_logi
print(diff.norm()/new_logits.norm())

tensor(3684.1467)
tensor(3680., dtype=torch.bfloat16)
tensor(7.0870)
tensor(0.0632)
tensor(0.0019)


In [8]:
calculator = api_cache.get_infomration_calculator(config.info_flow_model)
print("Original")
print(a:=calculator.calc_top_probabilities_from_logits(original_logi[-1],5))
print("New")
print(b:=calculator.calc_top_probabilities_from_logits(new_logits[-1],5))

Original
OrderedDict({' if': 0.392578125, ' how': 0.306640625, ' what': 0.1640625, ' where': 0.0220947265625, ' the': 0.013427734375})
New
OrderedDict({' if': 0.3714551329612732, ' how': 0.3197382986545563, ' what': 0.17012295126914978, ' where': 0.02258620224893093, ' the': 0.01313016377389431})
